# Semantic ID Generation — Part 2: Clustering & Quantization

**Based on**: *Transformer Memory as a Differentiable Search Index* (Tay et al., NeurIPS 2022),
*Product Quantization for Nearest Neighbor Search* (Jégou et al., TPAMI 2011)

---

Part 1 established the toy dataset, the semantic coherence metric, and showed that
random/atomic IDs provide no semantic structure. This part covers two approaches that
assign IDs based on embedding geometry:

- **Hierarchical K-Means** — recursive clustering; ID = tree path (§3)
- **Product Quantization** — independent subspace codebooks (§4)

We regenerate the same dataset (same seed) so results are directly comparable to Part 1.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from scipy.spatial.distance import cdist
from sklearn.cluster import KMeans

np.random.seed(42)

plt.rcParams.update({"figure.dpi": 110, "axes.grid": True, "grid.alpha": 0.3})

COLORS = ["steelblue", "darkorange", "seagreen", "crimson", "mediumpurple"]

---
## 1. Toy Dataset

We generate $N = 200$ items from 5 Gaussian clusters in two variants:

- **2D** ($D=2$): for scatter-plot visualisation throughout the notebook
- **64D** ($D=64$): for the quantisation approaches where higher dimensionality is needed

Both share the same cluster assignments so results are directly comparable.

### Semantic coherence metric

To quantify how well an ID scheme preserves neighbourhood structure, we define:

$$\text{SC}(f) = \frac{1}{N}\sum_{i=1}^{N} \frac{\bigl|\{j : j \in \text{kNN}_k(i) \;\land\; \text{prefix}_1(f(i)) = \text{prefix}_1(f(j))\}\bigr|}{k}$$

where $f$ maps items to IDs and $\text{kNN}_k(i)$ are the $k$ nearest neighbours by
embedding distance. $\text{prefix}_1$ denotes the first token of the ID.
SC $\in [0, 1]$: higher means neighbours share ID prefixes more often.

In [ ]:
N_ITEMS = 200
N_CLUSTERS = 5
ITEMS_PER_CLUSTER = N_ITEMS // N_CLUSTERS

# Cluster centres
centers_2d = np.array([
    [-3.0, -3.0],
    [ 3.0, -3.0],
    [ 0.0,  3.0],
    [-3.0,  3.0],
    [ 3.0,  3.0],
])

# Generate 2D embeddings
embeddings_2d = np.vstack([
    np.random.randn(ITEMS_PER_CLUSTER, 2) * 0.8 + centers_2d[idx]
    for idx in range(N_CLUSTERS)
])
labels = np.repeat(np.arange(N_CLUSTERS), ITEMS_PER_CLUSTER)

# Generate 64D embeddings with the same cluster structure
centers_64d = np.random.randn(N_CLUSTERS, 64) * 3.0
embeddings_64d = np.vstack([
    np.random.randn(ITEMS_PER_CLUSTER, 64) * 0.5 + centers_64d[idx]
    for idx in range(N_CLUSTERS)
])

print(f"2D embeddings : {embeddings_2d.shape}")
print(f"64D embeddings: {embeddings_64d.shape}")
print(f"Clusters      : {N_CLUSTERS}, {ITEMS_PER_CLUSTER} items each")

In [ ]:
def semantic_coherence(
    embeddings: np.ndarray,
    ids: np.ndarray,
    k: int = 10,
) -> float:
    """Fraction of k-NN that share the first token of their ID.

    Args:
        embeddings: (N, D) array of item embeddings.
        ids: (N,) or (N, L) array of IDs. First column used as prefix.
        k: number of nearest neighbours.

    Returns:
        SC in [0, 1].
    """
    ids = np.asarray(ids)
    if ids.ndim == 2:
        prefixes = ids[:, 0]
    else:
        prefixes = ids

    dists = cdist(embeddings, embeddings, metric="euclidean")
    np.fill_diagonal(dists, np.inf)
    n_items = len(embeddings)
    sc_sum = 0.0
    for idx in range(n_items):
        neighbours = np.argsort(dists[idx])[:k]
        matches = np.sum(prefixes[neighbours] == prefixes[idx])
        sc_sum += matches / k
    return sc_sum / n_items

In [ ]:
# Regenerate dataset and SC metric from Part 1 (same seed)

# --- Random IDs baseline (for comparison) ---
ids_random = np.arange(N_ITEMS)
np.random.shuffle(ids_random)
sc_random = semantic_coherence(embeddings_2d, ids_random)
sc_random_64d = semantic_coherence(embeddings_64d, np.random.permutation(N_ITEMS))

print(f"Dataset regenerated: {embeddings_2d.shape} (2D), {embeddings_64d.shape} (64D)")
print(f"Random ID baseline SC (2D): {sc_random:.3f}")

---
## 3. Approach 2: Hierarchical K-Means Clustering

Recursively partition the embedding space into $K$ clusters at each level.
An item's ID is the path from root to leaf:

$$\text{id}(i) = (c_1^{(i)}, c_2^{(i)}, \ldots, c_L^{(i)})$$

where $c_l^{(i)} \in \{0, \ldots, K-1\}$ is the cluster assignment at level $l$.
This is the approach used by DSI (Tay et al., NeurIPS 2022).

At each node $n$ with item set $\mathcal{S}_n$, run K-means:

$$\min_{\mu_1, \ldots, \mu_K} \sum_{i \in \mathcal{S}_n} \min_{k} \|\mathbf{e}_i - \mu_k\|^2$$

Recursion terminates when $|\mathcal{S}_n| \leq K$ or a maximum depth $L$ is reached.

The total vocabulary for the generative model is only $K \times L$ tokens instead of $N$:

$$|\text{vocab}| = K \cdot L \ll N$$

Items in the same coarse cluster share prefixes, enabling beam search to
generalise across similar items.

In [ ]:
def hierarchical_kmeans(
    embeddings: np.ndarray,
    K: int = 3,
    max_depth: int = 3,
) -> np.ndarray:
    """Assign hierarchical K-means IDs.

    Args:
        embeddings: (N, D) array.
        K: branching factor at each level.
        max_depth: maximum tree depth.

    Returns:
        ids: (N, max_depth) integer array of cluster assignments per level.
    """
    n_items = len(embeddings)
    ids = np.zeros((n_items, max_depth), dtype=np.int64)

    def _recurse(
        indices: np.ndarray, depth: int,
    ) -> None:
        if depth >= max_depth or len(indices) <= K:
            return
        km = KMeans(n_clusters=min(K, len(indices)), random_state=0, n_init=10)
        km.fit(embeddings[indices])
        for cluster_idx in range(km.n_clusters):
            child_mask = km.labels_ == cluster_idx
            child_indices = indices[child_mask]
            ids[child_indices, depth] = cluster_idx
            _recurse(child_indices, depth + 1)

    _recurse(np.arange(n_items), depth=0)
    return ids


hk_ids_2d = hierarchical_kmeans(embeddings_2d, K=5, max_depth=3)
print(f"ID shape: {hk_ids_2d.shape}")
print(f"Example IDs (first 5): {hk_ids_2d[:5].tolist()}")
print(f"Unique IDs: {len(np.unique(hk_ids_2d, axis=0))}")

In [ ]:
sc_hk_2d = semantic_coherence(embeddings_2d, hk_ids_2d)
print(f"SC (random IDs)            : {sc_random:.3f}")
print(f"SC (hierarchical K-means)  : {sc_hk_2d:.3f}")
print(f"Improvement                : {sc_hk_2d - sc_random:+.3f}")

### Exercise 2

Implement a single level of K-means clustering — the core building block of the
hierarchical approach. Given embeddings and a branching factor $K$, run K-means
and return the cluster assignments and centroids.

**Hint**: `KMeans(n_clusters=K, random_state=0, n_init=10).fit(embeddings)`

In [ ]:
def one_level_kmeans(
    embeddings: np.ndarray, K: int
) -> tuple[np.ndarray, np.ndarray]:
    """Run one level of K-means clustering.

    Args:
        embeddings: (N, D) array.
        K: number of clusters.

    Returns:
        assignments: (N,) array of cluster indices in {0, ..., K-1}.
        centers: (K, D) array of cluster centroids.
    """
    # YOUR CODE HERE: fit KMeans and extract labels + cluster_centers_
    pass

In [ ]:
test_assignments, test_centers = one_level_kmeans(embeddings_2d, K=5)
assert test_assignments.shape == (N_ITEMS,), "One assignment per item"
assert set(test_assignments).issubset(set(range(5))), "Labels must be in {0,...,K-1}"
assert test_centers.shape == (5, 2), "Must have K centers of dimension D"
print(f"Cluster sizes: {np.bincount(test_assignments)}")
print("Exercise 2 passed!")

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 4))
depth_labels = ["Level 0 (root)", "Level 1", "Level 2"]

for depth_idx in range(3):
    ax = axes[depth_idx]
    level_ids = hk_ids_2d[:, depth_idx]
    unique_ids = np.unique(level_ids)
    cmap = plt.cm.tab10
    for uid in unique_ids:
        mask = level_ids == uid
        ax.scatter(
            embeddings_2d[mask, 0], embeddings_2d[mask, 1],
            c=[cmap(uid / max(len(unique_ids), 1))], alpha=0.7, s=25,
            label=f"c={uid}",
        )
    ax.set_title(f"{depth_labels[depth_idx]} ({len(unique_ids)} groups)")
    ax.set_xlabel("$e_1$")
    if depth_idx == 0:
        ax.set_ylabel("$e_2$")

plt.suptitle("Hierarchical K-Means: deeper levels capture finer structure", y=1.02)
plt.tight_layout()
plt.show()

**Pros**: Simple and interpretable; vocabulary size $K \times L$ is small;
items in the same cluster share prefixes, enabling beam search generalisation.

**Cons**: Fixed tree structure with no overlap between branches;
sensitive to K-means initialisation; does not optimise for embedding reconstruction;
hard cluster boundaries — items near boundaries get very different IDs.

Hierarchical K-means encodes each item's position as a single path through a tree.
Product Quantisation takes a different approach: decompose the space into *independent
subspaces*, each with its own codebook.

In [ ]:
# Run hierarchical K-means on 64D too for fair comparison later
hk_ids_64d = hierarchical_kmeans(embeddings_64d, K=5, max_depth=3)
sc_hk_64d = semantic_coherence(embeddings_64d, hk_ids_64d)

sc_random_64d = semantic_coherence(
    embeddings_64d, np.random.permutation(N_ITEMS)
)

print(f"{'Approach':<25} {'SC (2D)':>8} {'SC (64D)':>8}")
print(f"{'Random IDs':<25} {sc_random:>8.3f} {sc_random_64d:>8.3f}")
print(f"{'Hierarchical K-Means':<25} {sc_hk_2d:>8.3f} {sc_hk_64d:>8.3f}")

---
## 4. Approach 3: Product Quantization (PQ)

Split the $D$-dimensional embedding into $M$ disjoint subspaces of dimension
$D/M$ each. Within each subspace $m$, learn a codebook
$\mathcal{C}_m = \{\mathbf{c}_{m,1}, \ldots, \mathbf{c}_{m,K}\}$ via K-means.
The item's ID is the tuple of $M$ codebook indices:

$$\text{id}(i) = \bigl(q_1(\mathbf{e}_i^{(1)}),\; \ldots,\; q_M(\mathbf{e}_i^{(M)})\bigr)$$

where $\mathbf{e}_i^{(m)} \in \mathbb{R}^{D/M}$ is the $m$-th sub-vector and:

$$q_m(\mathbf{x}) = \arg\min_{k \in \{1,\ldots,K\}} \|\mathbf{x} - \mathbf{c}_{m,k}\|^2$$

The reconstructed embedding is the concatenation of the selected centroids:

$$\hat{\mathbf{e}}_i = [\mathbf{c}_{1, q_1};\; \mathbf{c}_{2, q_2};\; \ldots;\; \mathbf{c}_{M, q_M}]$$

Reconstruction error:

$$\mathcal{L}_{\text{PQ}} = \frac{1}{N}\sum_{i=1}^{N} \|\mathbf{e}_i - \hat{\mathbf{e}}_i\|^2$$

This is the classic approach from Jégou et al. (TPAMI 2011), foundational to FAISS.

**Key difference from hierarchical K-means**: PQ's $M$ codebooks are *independent* —
the subspaces do not interact. This yields $K^M$ possible IDs from only $M \times K$ centroids.

In [ ]:
class ProductQuantizer:
    """Product Quantization: split embedding into M subspaces, K-means each."""

    def __init__(self, D: int, M: int, K: int) -> None:
        assert D % M == 0, f"D={D} must be divisible by M={M}"
        self.D = D
        self.M = M
        self.K = K
        self.sub_dim = D // M
        self.codebooks: list[np.ndarray] = []

    def fit(self, embeddings: np.ndarray) -> None:
        """Learn codebooks by running K-means on each subspace."""
        self.codebooks = []
        for m_idx in range(self.M):
            sub_vectors = embeddings[:, m_idx * self.sub_dim : (m_idx + 1) * self.sub_dim]
            km = KMeans(n_clusters=self.K, random_state=0, n_init=10)
            km.fit(sub_vectors)
            self.codebooks.append(km.cluster_centers_)

    def encode(self, embeddings: np.ndarray) -> np.ndarray:
        """Encode embeddings to (N, M) codebook indices."""
        n_items = len(embeddings)
        codes = np.zeros((n_items, self.M), dtype=np.int64)
        for m_idx in range(self.M):
            sub_vectors = embeddings[:, m_idx * self.sub_dim : (m_idx + 1) * self.sub_dim]
            dists = cdist(sub_vectors, self.codebooks[m_idx])
            codes[:, m_idx] = np.argmin(dists, axis=1)
        return codes

    def decode(self, codes: np.ndarray) -> np.ndarray:
        """Reconstruct embeddings from (N, M) codes."""
        n_items = len(codes)
        recon = np.zeros((n_items, self.D))
        for m_idx in range(self.M):
            recon[:, m_idx * self.sub_dim : (m_idx + 1) * self.sub_dim] = (
                self.codebooks[m_idx][codes[:, m_idx]]
            )
        return recon

    def reconstruction_error(self, embeddings: np.ndarray) -> float:
        """Mean squared reconstruction error."""
        codes = self.encode(embeddings)
        recon = self.decode(codes)
        return float(np.mean((embeddings - recon) ** 2))

In [ ]:
pq = ProductQuantizer(D=64, M=8, K=16)
pq.fit(embeddings_64d)
pq_codes = pq.encode(embeddings_64d)
pq_error = pq.reconstruction_error(embeddings_64d)

print(f"Codebook shape per subspace: ({pq.K}, {pq.sub_dim})")
print(f"ID shape: {pq_codes.shape}  (N={N_ITEMS}, M={pq.M})")
print(f"Example IDs (first 3): {pq_codes[:3].tolist()}")
print(f"Reconstruction MSE: {pq_error:.4f}")
print(f"Possible IDs: {pq.K}^{pq.M} = {pq.K ** pq.M:,}")

sc_pq_64d = semantic_coherence(embeddings_64d, pq_codes)
print(f"SC (64D): {sc_pq_64d:.3f}")

### Exercise 3

Implement the encoding step of Product Quantization. Given an embedding matrix
and a list of pre-trained codebooks (one per subspace), split each embedding into
$M$ sub-vectors and assign each to its nearest codebook entry.

**Hint**: for each subspace, compute pairwise distances between sub-vectors
and centroids, then take `argmin`.

In [ ]:
def pq_encode(
    embeddings: np.ndarray,
    codebooks: list[np.ndarray],
    M: int,
) -> np.ndarray:
    """Encode embeddings using pre-trained PQ codebooks.

    Args:
        embeddings: (N, D) array.
        codebooks: list of M arrays, each (K, D/M).
        M: number of subspaces.

    Returns:
        codes: (N, M) array of codebook indices.
    """
    N, D = embeddings.shape
    sub_dim = D // M
    codes = np.zeros((N, M), dtype=np.int64)
    for m_idx in range(M):
        sub_vectors = embeddings[:, m_idx * sub_dim : (m_idx + 1) * sub_dim]
        # YOUR CODE HERE: compute distances to codebooks[m_idx] and assign nearest
        # Hint: dists has shape (N, K); codes[:, m_idx] = argmin over K
        pass
    return codes

In [ ]:
test_pq_codes = pq_encode(embeddings_64d, pq.codebooks, M=8)
assert test_pq_codes.shape == (N_ITEMS, 8)
assert np.all(test_pq_codes >= 0) and np.all(test_pq_codes < 16)
np.testing.assert_array_equal(test_pq_codes, pq.encode(embeddings_64d))
print("Exercise 3 passed!")

In [ ]:
# PQ on 2D data: M=2 subspaces of 1D each — creates a grid structure
pq_2d = ProductQuantizer(D=2, M=2, K=4)
pq_2d.fit(embeddings_2d)
pq_codes_2d = pq_2d.encode(embeddings_2d)

fig, ax = plt.subplots(figsize=(6, 5))

# Color by first subspace code, marker by second
markers = ["o", "s", "^", "D"]
cmap = plt.cm.Set1
for c0 in range(4):
    for c1 in range(4):
        mask = (pq_codes_2d[:, 0] == c0) & (pq_codes_2d[:, 1] == c1)
        if mask.any():
            ax.scatter(
                embeddings_2d[mask, 0], embeddings_2d[mask, 1],
                c=[cmap(c0 / 4)], marker=markers[c1], alpha=0.7, s=40,
                edgecolors="white", linewidths=0.5,
            )

# Draw grid lines at codebook boundaries
for m_idx in range(2):
    boundaries = sorted(pq_2d.codebooks[m_idx].flatten())
    for bval in boundaries:
        if m_idx == 0:
            ax.axvline(bval, color="gray", alpha=0.3, linestyle="--")
        else:
            ax.axhline(bval, color="gray", alpha=0.3, linestyle="--")

ax.set_title("PQ on 2D: M=2 subspaces × K=4 (color=subspace 0, shape=subspace 1)")
ax.set_xlabel("$e_1$ (subspace 0)")
ax.set_ylabel("$e_2$ (subspace 1)")
plt.tight_layout()
plt.show()

In [ ]:
# Reconstruction error vs M and K
fig, axes = plt.subplots(1, 2, figsize=(11, 4))

# Vary M with K=16 fixed
m_values = [2, 4, 8, 16, 32]
errors_by_m = []
for m_val in m_values:
    pq_tmp = ProductQuantizer(D=64, M=m_val, K=16)
    pq_tmp.fit(embeddings_64d)
    errors_by_m.append(pq_tmp.reconstruction_error(embeddings_64d))

axes[0].bar(range(len(m_values)), errors_by_m, color="steelblue", edgecolor="white")
axes[0].set_xticks(range(len(m_values)))
axes[0].set_xticklabels([str(mv) for mv in m_values])
axes[0].set_xlabel("M (number of subspaces)")
axes[0].set_ylabel("Reconstruction MSE")
axes[0].set_title("K=16 fixed, varying M")

# Vary K with M=8 fixed
k_values = [4, 8, 16, 32, 64]
errors_by_k = []
for k_val in k_values:
    pq_tmp = ProductQuantizer(D=64, M=8, K=k_val)
    pq_tmp.fit(embeddings_64d)
    errors_by_k.append(pq_tmp.reconstruction_error(embeddings_64d))

axes[1].bar(range(len(k_values)), errors_by_k, color="darkorange", edgecolor="white")
axes[1].set_xticks(range(len(k_values)))
axes[1].set_xticklabels([str(kv) for kv in k_values])
axes[1].set_xlabel("K (codebook size)")
axes[1].set_ylabel("Reconstruction MSE")
axes[1].set_title("M=8 fixed, varying K")

plt.suptitle("PQ reconstruction error: storage/accuracy tradeoff", y=1.02)
plt.tight_layout()
plt.show()

**Pros**: Well-understood, extremely fast (FAISS), $K^M$ possible IDs from $M \times K$
parameters, excellent for approximate nearest neighbour search.

**Cons**: Subspaces are *independent* — no interaction between them, limiting expressiveness;
the split is axis-aligned, missing correlations across subspaces;
codebooks are fixed after fitting (not end-to-end learnable with a downstream task).

PQ decomposes the embedding into independent subspaces. What if instead of decomposing
spatially, we decomposed *residually* — quantising the error left by each successive
codebook? And what if the codebooks were learned end-to-end through a neural network?

In [ ]:
print(f"{'Approach':<25} {'SC (2D)':>8} {'SC (64D)':>8}")
print(f"{'-' * 43}")

pq_2d_full = ProductQuantizer(D=2, M=2, K=4)
pq_2d_full.fit(embeddings_2d)
sc_pq_2d = semantic_coherence(embeddings_2d, pq_2d_full.encode(embeddings_2d))

print(f"{'Random IDs':<25} {sc_random:>8.3f} {sc_random_64d:>8.3f}")
print(f"{'Hierarchical K-Means':<25} {sc_hk_2d:>8.3f} {sc_hk_64d:>8.3f}")
print(f"{'Product Quantization':<25} {sc_pq_2d:>8.3f} {sc_pq_64d:>8.3f}")

---
## Appendix: Exercise Solutions

Expand the cells below only after attempting the exercises.

In [ ]:
# Solution 2: one_level_kmeans
def one_level_kmeans_solution(
    embeddings: np.ndarray, K: int
) -> tuple[np.ndarray, np.ndarray]:
    km = KMeans(n_clusters=K, random_state=0, n_init=10).fit(embeddings)
    return km.labels_, km.cluster_centers_

print("Solution 2: km = KMeans(...).fit(embeddings); return km.labels_, km.cluster_centers_")

In [ ]:
# Solution 3: pq_encode
def pq_encode_solution(
    embeddings: np.ndarray, codebooks: list[np.ndarray], M: int,
) -> np.ndarray:
    N, D = embeddings.shape
    sub_dim = D // M
    codes = np.zeros((N, M), dtype=np.int64)
    for m_idx in range(M):
        sub_vectors = embeddings[:, m_idx * sub_dim : (m_idx + 1) * sub_dim]
        dists = cdist(sub_vectors, codebooks[m_idx])
        codes[:, m_idx] = np.argmin(dists, axis=1)
    return codes

print("Solution 3: dists = cdist(sub_vectors, codebooks[m_idx]); codes[:, m_idx] = argmin(dists, axis=1)")

---
**Next**: [Part 3 — Learned Quantization & Comparison](./part3-learned-quantization.ipynb) covers
RQ-VAE and the final comparison across all four approaches.